In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import datetime

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
iqair_path="/home/slow_data/Air_Quality/IQAir_air_quality.csv"

In [3]:
df_aqi = pd.read_csv(iqair_path)
df_aqi.head()

,timestamp,station_name,longitude,latitude,aqi,WHO_exposure,PM2.5 (µg/m³),PM10 (µg/m³),O3 (µg/m³),NO2 (µg/m³),SO2 (µg/m³),CO (µg/m³),condition,temperature (°),humidity (%),pressure,wind_speed (km/h),wind_direction
0,2025-04-17 01:00:00,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.8418,21.005200,167,15.6,78.1,215.0,62.8,9.6,8.2,NaN,Nhiều mây,23,86,1007,13.2,135
1,2025-04-17 01:00:00,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.7947,21.003100,154,12.1,60.3,204.5,10.2,2.5,6.5,2.0,Nhiều mây,23,86,1007,13.4,136
2,2025-04-17 01:00:00,Minh Khai - Bắc Từ Liêm,105.7400,21.050000,132,5.2,26.2,218.7,21.0,NaN,0.1,1.4,Mưa,23,84,1007,12.3,132
3,2025-04-17 01:00:00,Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK),107.0844,10.367976,83,5.2,26.2,66.6,73.5,3.9,6.2,0.1,Nhiều mây,27,83,1010,19.0,103
4,2025-04-17 01:00:00,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.3357,20.938100,144,10.6,53.0,147.4,48.2,1.0,1.3,2.2,Nhiều mây,21,88,1007,11.7,119


In [4]:
df_aqi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84739 entries, 0 to 84738
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   timestamp          84739 non-null  object 
 1   station_name       84739 non-null  object 
 2   longitude          84739 non-null  float64
 3   latitude           84739 non-null  float64
 4   aqi                84739 non-null  int64  
 5   WHO_exposure       84282 non-null  float64
 6   PM2.5 (µg/m³)      84282 non-null  float64
 7   PM10 (µg/m³)       57782 non-null  float64
 8   O3 (µg/m³)         42808 non-null  float64
 9   NO2 (µg/m³)        40379 non-null  float64
 10  SO2 (µg/m³)        50653 non-null  float64
 11  CO (µg/m³)         57403 non-null  float64
 12  condition          84739 non-null  object 
 13  temperature (°)    84739 non-null  int64  
 14  humidity (%)       84739 non-null  int64  
 15  pressure           84739 non-null  int64  
 16  wind_speed (km/h)  847

In [5]:
filtered_df_loc = df_aqi.loc[df_aqi['station_name'] == 'FPT']

In [6]:
filtered_df_loc.head()

,timestamp,station_name,longitude,latitude,aqi,WHO_exposure,PM2.5 (µg/m³),PM10 (µg/m³),O3 (µg/m³),NO2 (µg/m³),SO2 (µg/m³),CO (µg/m³),condition,temperature (°),humidity (%),pressure,wind_speed (km/h),wind_direction
6355,2025-05-15 21:00:00,FPT,106.8091,10.8416,53,2.0,10.0,NaN,NaN,NaN,NaN,NaN,Mưa,30,75,1010,12.5,142
6368,2025-05-15 22:00:00,FPT,106.8091,10.8416,56,2.4,12.0,NaN,NaN,NaN,NaN,NaN,Nhiều mây,30,77,1010,9.3,122
6381,2025-05-15 23:00:00,FPT,106.8091,10.8416,73,4.2,21.0,NaN,NaN,NaN,NaN,NaN,Nhiều mây,29,79,1010,8.2,116
6394,2025-05-16 00:00:00,FPT,106.8091,10.8416,86,5.6,28.0,NaN,NaN,NaN,NaN,NaN,Nhiều mây,28,80,1010,8.0,99
6407,2025-05-16 01:00:00,FPT,106.8091,10.8416,105,7.4,37.0,NaN,NaN,NaN,NaN,NaN,Nhiều mây,28,81,1009,7.2,90


--------

In [7]:
aod_path = "/home/slow_data/Air_Quality/AOD/station_aod/IQAir_stations/"

In [8]:
df_aod = pd.read_csv(aod_path+"FPT.csv")

In [9]:
df_aod.head()

,timestamp,AOT,Uncertainty,AE,QA_flag,SSA,RF
0,2025-01-01 07:00:00,NaN,NaN,NaN,6140.0,NaN,NaN
1,2025-01-01 07:10:00,NaN,NaN,NaN,6140.0,NaN,NaN
2,2025-01-01 07:20:00,NaN,NaN,NaN,6140.0,NaN,NaN
3,2025-01-01 07:30:00,NaN,NaN,NaN,6140.0,NaN,NaN
4,2025-01-01 07:40:00,NaN,NaN,NaN,6140.0,NaN,NaN


--------

# Process and Merge

In [10]:
# 1. Load Ground Data
df_aqi = pd.read_csv(iqair_path)

# Data Cleaning & Sorting
# Sort to ensure shift() works correctly
df_aqi = df_aqi.sort_values(by=['station_name', 'timestamp'])

# Handle frozen data (consecutive duplicates) and low values
target_columns = ['aqi', 'PM2.5 (µg/m³)']
for col in target_columns:
    # Group by station to avoid mixing data between different stations
    prev_values = df_aqi.groupby('station_name')[col].shift(1)
    
    # Identify rows where the value is identical to the previous one
    is_frozen = df_aqi[col] == prev_values
    df_aqi.loc[is_frozen, col] = np.nan

    # Handle Low Values
    if col == 'aqi':
        is_low = df_aqi[col] < 50
        df_aqi.loc[is_low, col] = np.nan

# Standardize Columns
METRICS = ['AQI', 'PM2.5', 'PM10']
METRIC_UNITS = {'AQI': 'Index', 'PM2.5': 'µg/m³', 'PM10': 'µg/m³'}

column_mapping = {
    'timestamp': 'Timestamp',
    'station_name': 'Name',
    'PM2.5 (µg/m³)': 'PM2.5',
    'PM10 (µg/m³)': 'PM10',
    'aqi': 'AQI'
}
df_aqi = df_aqi.rename(columns=column_mapping)
df_aqi['Timestamp'] = pd.to_datetime(df_aqi['Timestamp'])

# Force numeric
for metric in METRICS:
    if metric in df_aqi.columns:
        df_aqi[metric] = pd.to_numeric(df_aqi[metric], errors='coerce')

station_names = df_aqi['Name'].unique()
print(f"Loaded {len(station_names)} stations.")

Loaded 19 stations.


-------------------------

# Graph

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from scipy import stats 
from sklearn.linear_model import RANSACRegressor
from sklearn.metrics import r2_score

In [12]:
ROOT_OUTPUT = '/home/slow_data/Air_Quality/Output/'

(x,y) = (AOD,AQI)

In [13]:
output_dir = ROOT_OUTPUT + 'Output_RANSAC_IQAir'

In [14]:
# --- PROCESSING LOOP ---

print("Starting RANSAC Analysis...")

for station_name in station_names:
    # --- FILENAME MATCHING LOGIC ---
    possible_filenames = [
        f"{station_name}.csv",
        f"{station_name.replace(':', '')}.csv",
        f"{station_name.replace(':', '-')}.csv",
        f"{station_name.replace('/', '-')}.csv"
    ]

    aod_file_path = None

    for fname in possible_filenames:
        full_path = os.path.join(aod_path, fname)
        if os.path.exists(full_path):
            aod_file_path = full_path
            break

    if not aod_file_path:
        # print(f"No AOD file for {station_name}")
        continue

    # --- READ AOD DATA ---
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])

        # Filter AOD (No uncertainty threshold applied as requested)
        # Just ensure valid data
        df_aod_raw = df_aod_raw.dropna(subset=['AOT'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')

    except Exception as e:
        print(f"Error reading {aod_file_path}: {e}")
        continue

    # Get Ground Station Data
    df_station = df_aqi[df_aqi['Name'] == station_name].copy()
    if df_station.empty: 
        continue
    # print(f"Processing: {station_name}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        # Resample AOD to hourly to match ground data
        df_hourly_aod = df_aod_raw.resample('h').mean().dropna(subset=['AOT']).reset_index()

        # Merge Ground and AOD
        merged = pd.merge(df_station, df_hourly_aod, on='Timestamp', how='inner')

        # Filter valid pairs
        valid_data = merged[[metric, 'AOT']].dropna()

        # Require enough data points for RANSAC
        MIN_SAMPLES = 12
        if len(valid_data) < MIN_SAMPLES:
            # print(f"Skipping {station_name} - {metric}: only {len(valid_data)} points (need {MIN_SAMPLES})")
            continue

        X = valid_data['AOT'].values.reshape(-1, 1)
        y = valid_data[metric].values
       
        # --- RANSAC REGRESSION ---
        try:
            # Adjust min_samples if dataset is smaller than expected
            if len(X) < MIN_SAMPLES:
                # This case is already handled above, but as safety net:
                actual_min_samples = max(2, len(X) // 2)
            else:
                actual_min_samples = MIN_SAMPLES

            ransac = RANSACRegressor(
                min_samples=actual_min_samples, 
                random_state=68, 
                max_trials=2000, 
                stop_n_inliers=int(0.3 * X.shape[0]), 
                stop_score=0.8
            )
            ransac.fit(X, y)

            inlier_mask = ransac.inlier_mask_
            outlier_mask = np.logical_not(inlier_mask)

            # Predict lines
            line_X = np.array([X.min(), X.max()]).reshape(-1, 1)
            line_y_ransac = ransac.predict(line_X)  

            # Stats on INLIERS
            X_inliers = X[inlier_mask]
            y_inliers = y[inlier_mask]

            # Need at least 2 points for regression stats
            if len(X_inliers) < 2:
                print(f"Skipping {station_name} - {metric}: insufficient inliers ({len(X_inliers)})")
                continue

            # Get regression parameters from RANSAC estimator
            slope = ransac.estimator_.coef_[0]
            intercept = ransac.estimator_.intercept_
            
            # Calculate R and R² from inliers
            r_value = np.corrcoef(X_inliers.ravel(), y_inliers.ravel())[0, 1]
            r_squared = r_value**2
            
            # Calculate standard error manually
            y_pred = slope * X_inliers.ravel() + intercept
            residuals = y_inliers - y_pred
            n = len(X_inliers)
            # Standard error of the regression
            std_err = np.sqrt(np.sum(residuals**2) / (n - 2)) if n > 2 else np.nan

            # --- PLOTTING ---
            plt.figure(figsize=(8, 6))
           
            # Plot Outliers
            if np.sum(outlier_mask) > 0:
                plt.scatter(X[outlier_mask], y[outlier_mask], color='lightgray', marker='.', label='Outliers')

            # Plot Inliers
            plt.scatter(X[inlier_mask], y[inlier_mask], color='blue', marker='o', edgecolors='w', s=50, label='Inliers (RANSAC)')

            # Plot Line
            plt.plot(line_X, line_y_ransac, color='red', linewidth=2, label='RANSAC Fit')
            plt.title(f"RANSAC: {metric} vs AOT\n{station_name}")
            plt.xlabel("AOT (Satellite)")
            plt.ylabel(f"{metric} (Ground - {METRIC_UNITS[metric]})")
            plt.legend(loc='upper right')
            plt.grid(True, ls='--', alpha=0.5)

            # Stats Text (Based on Inliers)
            stats_text = '\n'.join((
                f'R (Inliers) = {r_value:.2f}',
                f'R² (Inliers) = {r_squared:.2f}',
                f'N (Inliers) = {np.sum(inlier_mask)}/{len(valid_data)}',
                f'y = {slope:.2f}x + {intercept:.2f}'
            ))

            plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes,
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

            # --- SAVE LOGIC ---
            # Folder: Root/Metric/
            save_dir = os.path.join(output_dir, metric)
            os.makedirs(save_dir, exist_ok=True)

            # Filename: StationName.png
            safe_name = "".join([c for c in station_name if c.isalnum() or c in (' ', '-', '_')]).strip()
            filename = f"{safe_name}.png"

            plt.savefig(os.path.join(save_dir, filename), dpi=100)
            plt.close()
           
            print(f"Saved {metric} for {station_name}")

        except Exception as e:
            print(f"RANSAC failed for {station_name} - {metric}: {e}")
            continue

print("Analysis Complete.")

Starting RANSAC Analysis...
Saved AQI for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved PM2.5 for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved PM10 for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved AQI for FPT
Saved PM2.5 for FPT
Saved AQI for HCM - FPT Thuduc
Saved PM2.5 for HCM - FPT Thuduc
Saved AQI for Hà Nội: Chi cục BVMT (KK)
Saved PM2.5 for Hà Nội: Chi cục BVMT (KK)
Saved PM10 for Hà Nội: Chi cục BVMT (KK)
Saved AQI for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM2.5 for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM10 for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM2.5 for Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Saved PM10 for Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Saved AQI for Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)
Saved PM2.5 for Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)
Saved

(x,y) = (AQI,AOT)

In [15]:
output_dir = ROOT_OUTPUT + 'Output_RANSAC_IQAir_Transpose'

In [16]:
# --- PROCESSING LOOP ---
print("Starting RANSAC Analysis (Swapped: AQI -> X, AOT -> y)...")

for station_name in station_names:
    # --- FILENAME MATCHING LOGIC ---
    possible_filenames = [
        f"{station_name}.csv",
        f"{station_name.replace(':', '')}.csv",
        f"{station_name.replace(':', '-')}.csv",
        f"{station_name.replace('/', '-')}.csv"
    ]
    
    aod_file_path = None
    for fname in possible_filenames:
        full_path = os.path.join(aod_path, fname)
        if os.path.exists(full_path):
            aod_file_path = full_path
            break
            
    if not aod_file_path:
        # print(f"No AOD file for {station_name}")
        continue

    # --- READ AOD DATA ---
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        
        # Filter AOD (No uncertainty threshold applied as requested)
        # Just ensure valid data
        df_aod_raw = df_aod_raw.dropna(subset=['AOT'])
        
        df_aod_raw = df_aod_raw.set_index('Timestamp')
    except Exception as e:
        print(f"Error reading {aod_file_path}: {e}")
        continue

    # Get Ground Station Data
    df_station = df_aqi[df_aqi['Name'] == station_name].copy()
    if df_station.empty: 
        continue
    
    # print(f"Processing: {station_name}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        # Resample AOD to hourly to match ground data
        df_hourly_aod = df_aod_raw.resample('h').mean().dropna(subset=['AOT']).reset_index()
        
        # Merge Ground and AOD
        merged = pd.merge(df_station, df_hourly_aod, on='Timestamp', how='inner')
        
        # Filter valid pairs
        valid_data = merged[[metric, 'AOT']].dropna()
        
        # Require enough data points for RANSAC
        MIN_SAMPLES = 12
        if len(valid_data) < MIN_SAMPLES:
            # print(f"Skipping {station_name} - {metric}: only {len(valid_data)} points (need {MIN_SAMPLES})")
            continue
        
        # --- SWAPPED ASSIGNMENT HERE ---
        # X is now Ground Data (AQI/metric)
        # y is now Satellite Data (AOT)
        X = valid_data[metric].values.reshape(-1, 1)
        y = valid_data['AOT'].values
        
        # --- RANSAC REGRESSION ---
        try:
            # Adjust min_samples if dataset is smaller than expected
            if len(X) < MIN_SAMPLES:
                # This case is already handled above, but as safety net:
                actual_min_samples = max(2, len(X) // 2)
            else:
                actual_min_samples = MIN_SAMPLES

            ransac = RANSACRegressor(
                min_samples=actual_min_samples,
                random_state=68,
                max_trials=2000,
                stop_n_inliers=int(0.3 * X.shape[0]),
                stop_score=0.8
            )
            ransac.fit(X, y)
            
            inlier_mask = ransac.inlier_mask_
            outlier_mask = np.logical_not(inlier_mask)
            
            # Predict lines
            line_X = np.array([X.min(), X.max()]).reshape(-1, 1)
            line_y_ransac = ransac.predict(line_X)
            
            # Stats on INLIERS
            X_inliers = X[inlier_mask]
            y_inliers = y[inlier_mask]
            
            # Need at least 2 points for regression stats
            if len(X_inliers) < 2:
                print(f"Skipping {station_name} - {metric}: insufficient inliers ({len(X_inliers)})")
                continue

            # Get regression parameters from RANSAC estimator
            slope = ransac.estimator_.coef_[0]
            intercept = ransac.estimator_.intercept_
            
            # Calculate R and R² from inliers
            r_value = np.corrcoef(X_inliers.ravel(), y_inliers.ravel())[0, 1]
            r_squared = r_value**2
            
            # Calculate standard error manually
            y_pred = slope * X_inliers.ravel() + intercept
            residuals = y_inliers - y_pred
            n = len(X_inliers)
            # Standard error of the regression
            std_err = np.sqrt(np.sum(residuals**2) / (n - 2)) if n > 2 else np.nan

            # --- PLOTTING ---
            plt.figure(figsize=(8, 6))
            
            # Plot Outliers
            if np.sum(outlier_mask) > 0:
                plt.scatter(X[outlier_mask], y[outlier_mask], color='lightgray', marker='.', label='Outliers')
            
            # Plot Inliers
            plt.scatter(X[inlier_mask], y[inlier_mask], color='blue', marker='o', edgecolors='w', s=50, label='Inliers (RANSAC)')
            
            # Plot Line
            plt.plot(line_X, line_y_ransac, color='red', linewidth=2, label='RANSAC Fit')
            
            # Update Titles and Labels for the Swap
            plt.title(f"RANSAC: AOT vs {metric}\n{station_name}")
            plt.xlabel(f"{metric} (Ground - {METRIC_UNITS[metric]})")  # X Label    
            plt.ylabel("AOT (Satellite)")                              # Y Label
            
            plt.legend(loc='upper right')
            plt.grid(True, ls='--', alpha=0.5)
            
            # Stats Text (Based on Inliers)
            stats_text = '\n'.join((
                f'R (Inliers) = {r_value:.2f}',
                f'R² (Inliers) = {r_squared:.2f}',
                f'N (Inliers) = {np.sum(inlier_mask)}/{len(valid_data)}',
                f'y = {slope:.2f}x + {intercept:.2f}'
            ))
            
            plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
            
            # --- SAVE LOGIC ---
            save_dir = os.path.join(output_dir, metric)
            os.makedirs(save_dir, exist_ok=True)
            
            safe_name = "".join([c for c in station_name if c.isalnum() or c in (' ', '-', '_')]).strip()
            filename = f"{safe_name}.png"
            
            plt.savefig(os.path.join(save_dir, filename), dpi=100)
            plt.close()
            
            print(f"Saved {metric} for {station_name}")
            
        except Exception as e:
            print(f"RANSAC failed for {station_name} - {metric}: {e}")
            continue

print("Analysis Complete.")

Starting RANSAC Analysis (Swapped: AQI -> X, AOT -> y)...
Saved AQI for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved PM2.5 for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved PM10 for Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến
Saved AQI for FPT
Saved PM2.5 for FPT
Saved AQI for HCM - FPT Thuduc
Saved PM2.5 for HCM - FPT Thuduc
Saved AQI for Hà Nội: Chi cục BVMT (KK)
Saved PM2.5 for Hà Nội: Chi cục BVMT (KK)
Saved PM10 for Hà Nội: Chi cục BVMT (KK)
Saved AQI for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM2.5 for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM10 for Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Saved PM2.5 for Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Saved PM10 for Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Saved AQI for Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)
Saved PM2.5 for Hà Nội: Đại Học Bách Khoa cổng Parab

## Try Ransac on global 

In [17]:
# # --- 2. AGGREGATE DATA FROM ALL STATIONS ---
# print("Aggregating data from all stations...")

# global_data_list = []

# for station_name in station_names:
#     # Match Filename
#     possible_filenames = [
#         f"{station_name}.csv",
#         f"{station_name.replace(':', '')}.csv",
#         f"{station_name.replace(':', '-')}.csv",
#         f"{station_name.replace('/', '-')}.csv"
#     ]
    
#     aod_file_path = None
#     for fname in possible_filenames:
#         full_path = os.path.join(aod_path, fname)
#         if os.path.exists(full_path):
#             aod_file_path = full_path
#             break
            
#     if not aod_file_path: continue

#     try:
#         # Load AOD
#         df_aod_raw = pd.read_csv(aod_file_path)
#         df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
#         df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
#         df_aod_raw = df_aod_raw.dropna(subset=['AOT']).set_index('Timestamp')
        
#         # Resample AOD to hourly
#         df_hourly_aod = df_aod_raw.resample('h').mean().dropna(subset=['AOT']).reset_index()
        
#         # Get Station Data
#         df_station = df_aqi[df_aqi['Name'] == station_name].copy()
#         if df_station.empty: continue
        
#         # Merge
#         merged = pd.merge(df_station, df_hourly_aod, on='Timestamp', how='inner')
        
#         # Keep track of station name for later splitting
#         merged['StationID'] = station_name
        
#         global_data_list.append(merged)
        
#     except Exception as e:
#         print(f"Error processing {station_name}: {e}")
#         continue

# if not global_data_list:
#     raise ValueError("No data found! Check paths.")

# df_global = pd.concat(global_data_list, ignore_index=True)
# print(f"Total Data Points Aggregated: {len(df_global)}")

In [18]:
# from sklearn.metrics import r2_score, mean_squared_error

In [19]:
# # --- 3. GLOBAL RANSAC & LOCAL TESTING ---

# for metric in METRICS:
#     # Filter data for this metric
#     df_metric = df_global[['StationID', 'AOT', metric]].dropna()
    
#     if df_metric.empty:
#         print(f"No data for {metric}")
#         continue
        
#     print(f"\n--- Processing Global {metric} ({len(df_metric)} points) ---")
    
#     X_global = df_metric['AOT'].values.reshape(-1, 1)
#     y_global = df_metric[metric].values
    
#     # --- A. FIT GLOBAL RANSAC ---
#     try:
#         # RANSAC with min_samples=10 and increased max_trials for robustness
#         ransac = RANSACRegressor(min_samples=10, max_trials=1000, random_state=42)
#         ransac.fit(X_global, y_global)
        
#         global_slope = ransac.estimator_.coef_[0]
#         global_intercept = ransac.estimator_.intercept_
        
#         # Get Inlier stats for the Global Plot
#         inlier_mask = ransac.inlier_mask_
#         outlier_mask = np.logical_not(inlier_mask)
#         X_inliers = X_global[inlier_mask]
#         y_inliers = y_global[inlier_mask]
        
#         # Calculate R value for global inliers using linregress (same as RANSAC's internal OLS)
#         slope_stat, intercept_stat, r_value, p_value, std_err = stats.linregress(X_inliers.ravel(), y_inliers.ravel())
        
#         # --- PLOT 1: GLOBAL FIT ---
#         plt.figure(figsize=(10, 8))
#         plt.scatter(X_global[outlier_mask], y_global[outlier_mask], color='lightgray', marker='.', alpha=0.5, label='Outliers')
#         plt.scatter(X_global[inlier_mask], y_global[inlier_mask], color='blue', marker='o', s=10, alpha=0.5, label='Inliers')
        
#         line_X = np.array([X_global.min(), X_global.max()]).reshape(-1, 1)
#         line_y = ransac.predict(line_X)
#         plt.plot(line_X, line_y, color='red', linewidth=3, label=f'Global RANSAC\n(y={global_slope:.2f}x+{global_intercept:.2f})')
        
#         plt.title(f"GLOBAL RANSAC Model: {metric} vs AOT\n(All Stations Combined)")
#         plt.xlabel("AOT")
#         plt.ylabel(metric)
#         plt.legend()
        
#         # Save Global Plot
#         save_dir = os.path.join(output_dir, metric)
#         os.makedirs(save_dir, exist_ok=True)
#         plt.savefig(os.path.join(save_dir, "Global_Fit_All_Stations.png"))
#         plt.close()
        
#         print(f"Global Model: y = {global_slope:.2f}x + {global_intercept:.2f}")
        
#     except Exception as e:
#         print(f"Global RANSAC failed for {metric}: {e}")
#         continue

#     # --- B. TEST GLOBAL LINE ON INDIVIDUAL STATIONS ---
#     unique_stations = df_metric['StationID'].unique()
    
#     for station in unique_stations:
#         df_station = df_metric[df_metric['StationID'] == station]
        
#         if len(df_station) < 5: continue # Skip if too few points
        
#         X_station = df_station['AOT'].values
#         y_station = df_station[metric].values
        
#         # Check for constant data to avoid errors
#         if np.std(X_station) == 0 or np.std(y_station) == 0:
#              continue
        
#         # PREDICT using GLOBAL parameters
#         y_pred = global_slope * X_station + global_intercept
        
#         # Calculate Stats (How well does the GLOBAL line fit THIS station?)
#         # Note: R2 can be negative here if the global model is worse than a horizontal line for this specific station.
#         r2 = r2_score(y_station, y_pred)
#         rmse = np.sqrt(mean_squared_error(y_station, y_pred))
#         n = len(y_station)
        
#         # --- PLOT STATION --- 
#         plt.figure(figsize=(7, 6))
#         plt.scatter(X_station, y_station, color='green', alpha=0.7, label='Station Data')
        
#         # Draw Global Line
#         line_x_sta = np.array([X_station.min(), X_station.max()])
#         line_y_sta = global_slope * line_x_sta + global_intercept
#         plt.plot(line_x_sta, line_y_sta, color='red', linewidth=2, linestyle='-', label='Global Model')
        
#         plt.title(f"{station}\nTested against Global {metric} Model")
#         plt.xlabel("AOT")
#         plt.ylabel(metric)
#         plt.grid(True, ls='--', alpha=0.5)
#         plt.legend()
        
#         stats_text = '\n'.join((
#             f"Global Slope: {global_slope:.2f}",
#             f"Global Intercept: {global_intercept:.2f}",
#             f"Local R²: {r2:.2f}",
#             f"Local RMSE: {rmse:.2f}",
#             f"N: {n}"
#         ))
        
#         plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
#                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
#         safe_name = "".join([c for c in station if c.isalnum() or c in (' ', '-', '_')]).strip()
#         plt.savefig(os.path.join(save_dir, f"{safe_name}.png"))
#         plt.close()

# print("Global Analysis Complete.")